In [6]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup as bs
from dbio3 import to_db, load_data

# 뉴스목록, 날짜, id수집

In [7]:
url1 = "https://fintech.or.kr/web/board/boardContentsList.do"

In [8]:
payload = dict(miv_pageNo=1, mode='W', board_id=6)

In [9]:
r1 = requests.get(url1, params=payload)
print(r1.status_code)
soup = bs(r1.content, "lxml")
soup

200


<html><head><script language="javascript">
$(document).ready(function(){

	// 댓글수 노출여부 확인후 숨기기(미구현)
	
		$(".comment_cnt").hide();
	

	// 답변 노출여부 확인후 숨기기(미구현)
	
		$(".reply_status").hide();
	

	// 모바일 전용 페이지 숨기지
	//$(".mobile_list").hide();

	$(function() {

	});

	$("#searchtxt").keydown(function(key) {
		if (key.keyCode == 13) {
			search();
		}
	});
});
</script>
</head><body><div class="boardlist_top">
<ul class="box">
<li class="select">
<div class="optionbox">
<select id="searchkey" name="searchkey">
<option value="I">전체</option>
<option value="T">제목</option>
<option value="C">내용</option>
</select>
</div>
</li>
<li class="input">
<div class="inpbox"><input class="txt" id="searchtxt" name="searchtxt" placeholder="검색어 입력" title="검색어 입력" type="text" value=""/></div>
</li>
<li class="button">
<button class="btn3 bg_blue" onclick="search();" title="검색" type="button">검색</button>
<!-- <button type="button" class="btn3 bg_gray" title="재검색">재검색</button> -->
</li>
</ul>
</div>
<!--// list_t

mobile용

In [10]:
len(soup.select("ul.list > li > a"))

15

Web용은 Table 안에

In [11]:
# 제목과 뉴스 id 
soup.select("a.txtl")

[<a class="txtl" href="javascript:contentsView('961c6a72b1d0486e86711b77872a7471')">[세종테크노파크] 2025년 세종테크밸리 첨단기업 유치 임차료 지원사업 희망기업(임차기업) 모집 공고</a>,
 <a class="txtl" href="javascript:contentsView('b4b0b17c318d45158645c81c1472e29a')">[금융보안원] 「데이터허브」 이용 안내</a>,
 <a class="txtl" href="javascript:contentsView('51cb0e37c4064f2a985436e2ae3378df')">[국무조정실] 규제개혁신문고 제도 안내</a>,
 <a class="txtl" href="javascript:contentsView('3c24d05040984c79a7cdfd2c921952eb')">[기술보증기금] 핀테크 기업을 위한 주요 보증상품 안내</a>,
 <a class="txtl" href="javascript:contentsView('25f4360d5b9847a996828099e8d52ade')">[신용보증기금] 스타트업 지원 업무 안내</a>,
 <a class="txtl" href="javascript:contentsView('9c16609a99ad4c55914ffa9a7f2d58b7')">[한국무역보험공사] K-Sure 수출컨설팅 사업 안내</a>,
 <a class="txtl" href="javascript:contentsView('6b14460b5adc480aa050c4c535af8b72')">[한국예탁결제원] 예탁결제원 보유 데이터 활용 안내</a>,
 <a class="txtl" href="javascript:contentsView('d4cbbeb3f9b3444cbf1856b48f9f2926')">[웰컴저축은행] 마이데이터 서비스 제휴 솔루션</a>,
 <a class="txtl" href="javascript:contentsView('

web용은 Table 안에

In [12]:
# 제목과 뉴스 id 
soup.select("a.txtl")[0]['href'].split("'")[1]

'961c6a72b1d0486e86711b77872a7471'

In [13]:
# 날짜
soup.select("td.last")[0].text.strip()

'2025-12-08'

# news list id와 날짜 모으기

In [14]:
contents_id_list = []
for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
    date = date.text.strip()
    contents_id = content_list['href'].split("'")[1]
    contents_id_list.append((date, contents_id))
contents_id_list

[('2025-12-08', '961c6a72b1d0486e86711b77872a7471'),
 ('2025-09-17', 'b4b0b17c318d45158645c81c1472e29a'),
 ('2024-02-20', '51cb0e37c4064f2a985436e2ae3378df'),
 ('2022-12-19', '3c24d05040984c79a7cdfd2c921952eb'),
 ('2022-12-19', '25f4360d5b9847a996828099e8d52ade'),
 ('2022-06-20', '9c16609a99ad4c55914ffa9a7f2d58b7'),
 ('2021-12-03', '6b14460b5adc480aa050c4c535af8b72'),
 ('2021-04-07', 'd4cbbeb3f9b3444cbf1856b48f9f2926'),
 ('2025-12-09', '31c875b19d884ba5aab60d1cfae1411a'),
 ('2025-12-08', '6e8f6c4a35394b3f851c156b71ca33b7'),
 ('2025-12-05', '796323edef8b4ca79c0a2bbf17a262c3'),
 ('2025-12-04', '778f539d67bd409daaaba7ad7dc09aa5'),
 ('2025-12-03', '9bc42721d9a84e6295893e5b0ee07581'),
 ('2025-12-02', 'c23ab5f950854287905dfeb7f6f94664'),
 ('2025-12-01', 'ca4e219afc7747ec941b31d5eff7cb1f')]

In [15]:
# newslist의 url
url2 = "https://fintech.or.kr/web/board/boardContentsView.do"

In [16]:
# news list의 payload
payload2 = dict(miv_pageNo=1, mode="W", contents_id="39547f2a0ee34a5097ff4e92f6df9f6e", board_id=6, searchkey="I")

In [17]:
r2 = requests.get(url2, params=payload2)
print(r2.status_code)
soup2 = bs(r2.content,'lxml')
soup2

200


<!DOCTYPE html>
<html lang="ko" xml:lang="ko">
<head>
<title>
		
		
		핀테크 포털 - 한국핀테크지원센터
		
		</title>
<meta content="핀테크 생태계 활성화로 금융의 혁신과 성장을 지원합니다.." name="description"/>
<meta content="핀테크 포털 - 한국핀테크지원센터" name="keywords"/>
<meta content="http://fintech.or.kr/images/preview_page.png" property="og:image"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="width=device-width,initial-scale=1.0,maximum-scale=1.0,minimum-scale=1.0,user-scalable=no" name="viewport"/>
<meta content="XpressEngine" name="Generator"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="o8yb702i79l2intg1e9ov2hyavzbji" name="facebook-domain-verification"/>
<!-- <link rel="shortcut icon" href="/images/web/favicon.ico" type="image/x-icon"> -->
<!-- <link rel="icon" href="/images/web/favicon.ico" type="image/x-icon"> -->
<link href="/css/web/default.css?ver=20200916" rel="stylesheet" type="text/css"/>
<link href="/css/web/reset.css?ver=20210629" rel="stylesheet" t

In [18]:
# 뉴스 리스트 목록
soup2.select("tbody td a")[0].text

"[한국핀테크지원센터X구름] K-디지털트레이닝 '핀테크 인턴십 코스' 4기 훈련생 모집(~12.8.)"

In [19]:
soup2.select("tbody td a")[0]['href']

'https://fintech.or.kr/web/board/boardContentsView.do?board_id=3&contents_id=ba9a817cc8434910a4c6f47ff2e19099&menu_id=6300'

In [20]:
soup2.select("tbody td a")[6]['href']

'https://n.news.naver.com/mnews/article/003/0013620831?sid=101'

# 위에서 개별 작업한 코드 모으기

In [21]:
# 핀테크 뉴스 리스트, 날짜, content_id 수집
contents_id_list = []
for page in range(1, 20):
    url1 = "https://fintech.or.kr/web/board/boardContentsList.do"
    payload = dict(miv_pageNo=page, mode='W', board_id=6)
    r1 = requests.post(url1, data=payload)
    print(r1.status_code)
    soup = bs(r1.content, "lxml")

    for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
        date = date.text.strip()
        contents_id = content_list['href'].split("'")[1]
        contents_id_list.append((date, contents_id))
    time.sleep(2)

result = {}
# 날짜별 뉴스 리스트의 세부 뉴스 제목, 링크
for idx, (date, content_id) in enumerate(contents_id_list):
    print(f"{idx+1}/{len(contents_id_list)} 수집중", end="\r")
    url2 = "https://fintech.or.kr/web/board/boardContentsView.do"
    payload2 = dict(miv_pageNo=1, mode="W", contents_id=content_id, board_id=6, searchkey="I")
    r2 = requests.get(url2, params=payload2)
#     print(r2.status_code)
    soup2 = bs(r2.content,'lxml')

    for td in soup2.select("tbody td a"): 
        if "https://n.news.naver.com" in td['href'] or "https://fintech.or.kr/web/" in td['href']:
            result.setdefault('date', []).append(date)
            result.setdefault('제목', []).append(td.text)
            result.setdefault('뉴스링크', []).append(td['href'])
            
df = pd.DataFrame(result)
df

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


,date,제목,뉴스링크
0,2025-12-09,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...
1,2025-12-09,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...
2,2025-12-09,"금융위, 새도약기금 소각식…""취약층 연체채권 1.1조 소각""",https://n.news.naver.com/mnews/article/003/001...
3,2025-12-09,한·일 금융감독 정례회의…국제금융협력 포럼도 개최,https://n.news.naver.com/mnews/article/277/000...
4,2025-12-09,"해외투자 급증에… 금감원, 키움·하나증권 영업 실태 점검",https://n.news.naver.com/mnews/article/366/000...
...,...,...,...
11702,2025-01-02,"""당장 돈 안되고 성능향상 기대 못 미쳐도""… 세계는 AI인프라 `영끌`",https://n.news.naver.com/mnews/article/029/000...
11703,2025-01-02,트럼프 2.0시대 중국…미 동맹국 이탈 기대하며 버티기[다시 만난 트럼프②],https://n.news.naver.com/mnews/article/032/000...
11704,2025-01-02,'美 51번째 주' 모욕 당했는데…트럼프에 찍소리 못하는 이유 [김리안의 에네르기파...,https://n.news.naver.com/mnews/article/015/000...
11705,2025-01-02,[산업은행] 2025년 상반기 KDB NextONE 참여 스타트업 모집(서울/부산)...,https://fintech.or.kr/web/board/boardContentsV...


# 세부링크로 들어가서 뉴스 본문 수집

In [22]:
df2 = df[df['date'] > "2025-01-01"].copy()

In [23]:
df2['뉴스링크'][6]

'https://n.news.naver.com/mnews/article/119/0003034561?sid=101'

In [24]:
# 핀테크 지원센터 글
r3 = requests.get(df2['뉴스링크'][0])
soup3 = bs(r3.content, 'lxml')
soup3.select_one("div.content").text

'안녕하세요.\xa0한국핀테크지원센터입니다.2025년 금융특화 계층별\xa0AI역량강화 과정\xa0‘핀테크를 통한 금융\xa0AI\xa0트렌드와 혁신 사례’이 아래 일정으로 진행됩니다.많은 관심과 참여 바랍니다.\xa0□\xa0(교육대상)\xa0금융·인공지능에 관심 있는 누구나\xa0□\xa0(교육기간)\xa0~ 2025.12.31.(수)\xa0상시\xa0□\xa0(교육방식)\xa0온라인 수강(https://fintech.ekma.co.kr)\xa0□\xa0(교 육 비)\xa0전액지원\xa0□\xa0(교육혜택)\xa0수강 과정 코스별 한국핀테크지원센터 수료증 발급\xa0□\xa0(커리큘럼)\xa0차수강의명강사1핀테크 비즈니스 현황 및 트렌드SK증권 서영수 상무2핀테크 비즈니스\xa0AI\xa0활용 현황자본시장연구원 이성복 선임연구위원3핀테크 비즈니스\xa0AI\xa0혁신 사례한국오라클 김현웅 본부장4금융분야의\xa0AI\xa0도입을 위한 규제법무법인 덕수 신하나 변호사5간편결제 사례한국핀테크산업협회 이근주 회장6자산관리 사례자본시장연구원 이성복 선임연구위원7여신 분야\xa0AI\xa0사례길진세 한국금융연수원 강사8은행 분야\xa0AI\xa0사례길진세 한국금융연수원 강사9인슈어테크 사례1네이버 최욱동 리더10인슈어테크 사례2네이버 최욱동 리더👉\xa0교육신청 바로가기'

In [25]:
# 네이버뉴스 글
r3 = requests.get(df2['뉴스링크'][6])
soup3 = bs(r3.content, 'lxml')
soup3.select_one("#dic_area").text

'\n금융상품·부채관리 등 실생활 금융교육대학생 금융 이해력 제고\n\n\n\n금융감독원이 전국 대학을 대상으로 2026년 1학기 ‘실용금융’ 강좌 개설 지원 신청을 받는다.ⓒ연합뉴스[데일리안 = 김민환 기자] 금융감독원은 전국 대학을 대상으로 2026년 1학기 ‘실용금융’ 강좌 개설 지원 신청을 받는다고 9일 밝혔다. 오는 31일까지 신청을 접수해 강좌를 개설하는 대학에 금융교육 교수 인력과 교재, 온라인 강좌 자료 등을 지원할 예정이다.‘실용금융’은 금융상품 이해, 부채 및 신용관리, 연금, 보험, 금융소비자 보호 제도 등 실생활에 필요한 금융지식을 다루는 강좌로, 금감원은 2016년부터 해당 강좌를 개설하는 대학을 대상으로 지원 사업을 운영해 왔다.올해 2학기에는 전국 65개 대학에서 76개 실용금융 강좌가 개설돼 약 4748명이 수강 중이며, 수강생을 대상으로 한 설문조사에서 금융 이해도 향상과 강의 만족도가 높은 것으로 나타났다.금감원은 대학의 방식에 따라 금융 관련 현장 경험을 갖춘 금감원 직원의 직접 출강, 학생용 교재 제공, 교수 보조자료 지원, 정규 온라인 강좌용 강의 영상과 학습 자료 제공 등 다양한 형태로 지원할 계획이다.신청은 금감원 e-금융교육센터 홈페이지를 통해 온라인으로 가능하며, 기타문의는 금감원 금융교육국을 통해 안내받을 수 있다.\n\t\t'

In [26]:
df2['뉴스본문'] = ""
df2

,date,제목,뉴스링크,뉴스본문
0,2025-12-09,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...,
1,2025-12-09,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...,
2,2025-12-09,"금융위, 새도약기금 소각식…""취약층 연체채권 1.1조 소각""",https://n.news.naver.com/mnews/article/003/001...,
3,2025-12-09,한·일 금융감독 정례회의…국제금융협력 포럼도 개최,https://n.news.naver.com/mnews/article/277/000...,
4,2025-12-09,"해외투자 급증에… 금감원, 키움·하나증권 영업 실태 점검",https://n.news.naver.com/mnews/article/366/000...,
...,...,...,...,...
11702,2025-01-02,"""당장 돈 안되고 성능향상 기대 못 미쳐도""… 세계는 AI인프라 `영끌`",https://n.news.naver.com/mnews/article/029/000...,
11703,2025-01-02,트럼프 2.0시대 중국…미 동맹국 이탈 기대하며 버티기[다시 만난 트럼프②],https://n.news.naver.com/mnews/article/032/000...,
11704,2025-01-02,'美 51번째 주' 모욕 당했는데…트럼프에 찍소리 못하는 이유 [김리안의 에네르기파...,https://n.news.naver.com/mnews/article/015/000...,
11705,2025-01-02,[산업은행] 2025년 상반기 KDB NextONE 참여 스타트업 모집(서울/부산)...,https://fintech.or.kr/web/board/boardContentsV...,


In [ ]:
for idx, link in enumerate(df2['뉴스링크'][:10]):
    print(f"{idx+1}/{len(df2['뉴스링크'][:10])} 수집중", end="\r")
    r4 = requests.get(link)
    soup4 = bs(r4.content, 'lxml')
    if "https://fintech.or.kr/web/" in link:
        r3 = requests.get(link)
        soup3 = bs(r3.content, 'lxml')
        article = soup3.select_one("div.content").text
        df2.loc[idx, '뉴스본문'] = article
    elif "https://n.news.naver.com" in link:
        r3 = requests.get(link)
        soup3 = bs(r3.content, 'lxml')
        article = soup3.select_one("#dic_area").text
        df2.loc[idx, '뉴스본문'] = article
    to_db("fintech_news", "news_articles", df2.loc[[idx], :])
    time.sleep(3)
df2

In [27]:
df2['뉴스본문']

0         
1         
2         
3         
4         
        ..
11702     
11703     
11704     
11705     
11706     
Name: 뉴스본문, Length: 11707, dtype: object

In [28]:
df2.to_csv("./data/fintech_news.csv")

In [29]:
df2 = pd.read_csv("./data/fintech_news.csv", index_col=0)
df2

,date,제목,뉴스링크,뉴스본문
0,2025-12-09,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...,NaN
1,2025-12-09,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...,NaN
2,2025-12-09,"금융위, 새도약기금 소각식…""취약층 연체채권 1.1조 소각""",https://n.news.naver.com/mnews/article/003/001...,NaN
3,2025-12-09,한·일 금융감독 정례회의…국제금융협력 포럼도 개최,https://n.news.naver.com/mnews/article/277/000...,NaN
4,2025-12-09,"해외투자 급증에… 금감원, 키움·하나증권 영업 실태 점검",https://n.news.naver.com/mnews/article/366/000...,NaN
...,...,...,...,...
11702,2025-01-02,"""당장 돈 안되고 성능향상 기대 못 미쳐도""… 세계는 AI인프라 `영끌`",https://n.news.naver.com/mnews/article/029/000...,NaN
11703,2025-01-02,트럼프 2.0시대 중국…미 동맹국 이탈 기대하며 버티기[다시 만난 트럼프②],https://n.news.naver.com/mnews/article/032/000...,NaN
11704,2025-01-02,'美 51번째 주' 모욕 당했는데…트럼프에 찍소리 못하는 이유 [김리안의 에네르기파...,https://n.news.naver.com/mnews/article/015/000...,NaN
11705,2025-01-02,[산업은행] 2025년 상반기 KDB NextONE 참여 스타트업 모집(서울/부산)...,https://fintech.or.kr/web/board/boardContentsV...,NaN


In [32]:
# !pip install sqlalchemy pymysql

In [33]:
# !pip install python-dotenv

In [ ]:
df3 = load_data("fintech_news", "news_articles")
df3